In [23]:
from pathlib import Path
import os
import shutil
import zipfile

from urllib.request import urlretrieve

import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image


In [ ]:
# ========================================================
# 下載資料集
# ========================================================
DATA_ROOT = Path("./data")
DATA_ROOT.mkdir(exist_ok=True)

TIN_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
TIN_ZIP = DATA_ROOT / "tiny-imagenet-200.zip"
OUT_DIR = DATA_ROOT / "tiny-3class"

# class mapping: {wnid: (class_name, max_images)}
CLASS_INFO = {
    "n02099712": ("Labrador_Retriever", 400),
    "n02504458": ("African_Elephant", 400),
    "n01443537": ("Goldfish", 200),
}

# ========================================================
# Step 1 — Download tiny-imagenet zip
# ========================================================
if not OUT_DIR.exists():
    if not TIN_ZIP.exists():
        print("Downloading Tiny-ImageNet...")
        urlretrieve(TIN_URL, TIN_ZIP)
        print("Download complete.")
    else:
        print("Zip already exists.")
    
# ========================================================
# Step 2 — Selective extraction
# ========================================================

    print("\nExtracting ONLY selected categories...\n")

    with zipfile.ZipFile(TIN_ZIP, "r") as z:

        for wnid, (class_name, max_count) in CLASS_INFO.items():

            # output folder for images
            class_folder = OUT_DIR / class_name
            class_folder.mkdir(parents=True, exist_ok=True)

            prefix = f"tiny-imagenet-200/train/{wnid}/images/"
            img_files = sorted([
                f for f in z.namelist()
                if f.startswith(prefix) and f.lower().endswith(".jpeg")
            ])

            img_files = img_files[:max_count]  # limit number

            # extract limited images
            for f in img_files:
                z.extract(f, OUT_DIR)
                extracted = OUT_DIR / f
                # move to final location
                shutil.move(str(extracted), str(class_folder / Path(f).name))

            print(f"✔ {class_name}: {len(img_files)} images extracted")

            # remove intermediate tiny-imagenet-200 folder if exists
            temp_root = OUT_DIR / "tiny-imagenet-200"
            if temp_root.exists():
                shutil.rmtree(temp_root)
    print("\n Extraction complete!")
    print(f"Dataset ready at: {OUT_DIR}")

Zip already exists.

Extracting ONLY selected categories...

✔ Labrador_Retriever: 400 images extracted
✔ African_Elephant: 400 images extracted
✔ Goldfish: 200 images extracted

 Extraction complete!
Dataset ready at: data\tiny-3class


In [ ]:
# ========================================================
# resized to 64 × 64
# ========================================================

# 1. 定義 transform：Resize + ToTensor + Normalize
#    Resize 到 64x64
transform = transforms.Compose([
    transforms.Resize((64, 64)),          # 題目要求的大小
    transforms.ToTensor(),                # 轉成 [0,1] tensor，形狀 C×H×W
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],       # 常用 ImageNet mean
        std=[0.229, 0.224, 0.225]         # 常用 ImageNet std
    ),
])

# 2. 用 ImageFolder 建立 Dataset
#   - root 底下每個子資料夾視為一個 class
#   - 我們現在是：tiny-3class/Labrador_Retriever/images/*.JPEG
#     ImageFolder 會遞迴往下找圖片，所以 OK
dataset = datasets.ImageFolder(
    root=str(OUT_DIR),
    transform=transform
)

# # 3. 看一下基本資訊
# print("Total samples:", len(dataset))
# print("Classes (folder name → label index):")
# print(dataset.class_to_idx)

# # 4. 建一個 DataLoader 測試一下
# loader = DataLoader(dataset, batch_size=32, shuffle=True)

# # 取一個 batch 看形狀
# images, labels = next(iter(loader))
# print("Batch image shape:", images.shape)   # [32, 3, 64, 64]
# print("Batch labels:", labels[:10])

Total samples: 1000
Classes (folder name → label index):
{'African_Elephant': 0, 'Goldfish': 1, 'Labrador_Retriever': 2}
Batch image shape: torch.Size([32, 3, 64, 64])
Batch labels: tensor([0, 0, 0, 1, 0, 0, 0, 2, 1, 2])


In [22]:
# ===== Train/Val split =====

# 設定隨機種子，確保每次 split 一樣（可重現）
torch.manual_seed(42)

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print("Total samples:", dataset_size)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

# 建立 DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 測試一個 batch
train_images, train_labels = next(iter(train_loader))
print("Train batch shape:", train_images.shape)
print("Train batch labels:", train_labels[:10])

Total samples: 1000
Train samples: 800
Val samples: 200
Train batch shape: torch.Size([32, 3, 64, 64])
Train batch labels: tensor([2, 2, 2, 0, 2, 2, 0, 0, 2, 2])


In [ ]:
class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()

        # Convolution blocks
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)

        # After 3 maxpools → 64 × 8 × 8
        self.fc1 = nn.Linear(64 * 8 * 8, 128)   # latent feature (128-D)
        self.fc2 = nn.Linear(128, 3)            # 3 classes

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

        # Block 2
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        # Block 3
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)

        # Flatten
        x = x.view(x.size(0), -1)

        # FC layers
        latent = F.relu(self.fc1(x))
        out = self.fc2(latent)

        return out, latent